# ChiNeutro: mT0 / mT5 ajustados en Reddit Chile (evaluación fuera de dominio)

Traduce las 1.900 frases de `reddit_chile.json` con los dos modelos **ya entrenados** (`mt0_large_lora_r16` y
`mt5_large_lora_r16`, LoRA r16 de `models/mt0_vs_mt5/entrenar_mt5.ipynb`) y calcula las métricas automáticas.
No vuelve a entrenar: carga el adaptador que quedó en Drive (`MyDrive/chineutro_mt5/<NOMBRE>/modelo_final`).

Mismo procedimiento que Gemma 4 en Reddit (`models/gemma4/predecir_reddit.py`): greedy, mismo prompt que en
entrenamiento (`INSTRUCCION_CHAT`), primera línea, y `evaluar(..., juez=False)`. Sin referencia, BERTScore se calcula
**contra la entrada**, así que no se compara con el de test.

**Cómo usarlo**
1. Entorno de ejecución → Cambiar tipo → **T4 GPU** (sirve cualquier GPU).
2. Sube `data/evaluacion/reddit_chile.json` (de tu computador) a `MyDrive/chineutro_mt5/` en Google Drive. Si no
   está ahí, el notebook te pide subirlo y guarda una copia en Drive para las siguientes ejecuciones. El archivo no
   está en GitHub (está en `.gitignore`), así que el repo clonado no lo trae. No necesitas token de Hugging Face.
3. Ejecutar todo. Las predicciones quedan en Drive (`MyDrive/chineutro_mt5/<NOMBRE>/predicciones/reddit/`) y al final
   se descarga un zip para copiar a `models/mt0_vs_mt5/predicciones/reddit/` en tu computador.
4. El juez Opus se corre después en tu computador con `models/gemma4/juzgar_reddit.py` (ver la última celda).

Si Colab se desconecta, vuelve a ejecutar todo: los modelos que ya tienen su CSV en Drive no se traducen de nuevo.

In [ ]:
%pip install -q -U "transformers>=4.56" "peft>=0.13" accelerate sacrebleu bert-score==0.3.13
%pip install -q sentencepiece "protobuf<7"
%pip uninstall -q -y torchao   # el torchao de Colab rompe a peft; no se usa

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución -> Cambiar tipo -> T4 GPU"
p = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.get_device_capability(0)[0] >= 8      # A100/L4 sí, T4 no
print(f"GPU: {p.name} | {p.total_memory / 1e9:.0f} GB | bf16 nativo: {BF16}")

## Configuración

In [ ]:
# Corridas ya entrenadas (nombre de la carpeta en Drive -> modelo base)
MODELOS = {
    "mt0_large_lora_r16": "bigscience/mt0-large",
    "mt5_large_lora_r16": "google/mt5-large",
}

# Generación: igual que en val/test (greedy, primera línea). Los largos máximos suben respecto del entrenamiento
# (128 / 96) porque las frases de Reddit llegan a 40 palabras y traen ruido; con frases cortas no cambia nada.
MAX_LEN_IN, MAX_NEW_TOKENS, BS = 256, 160, 32

LIMITE = None        # p. ej. 32 para una prueba de humo; None = las 1.900 frases
REHACER = False      # True para volver a traducir aunque el CSV ya esté en Drive

REPO_URL = "https://github.com/matiaraos/NLP-ChiNeutro.git"
RAMA = "main"                                   # tiene evaluacion/, comun.py y las predicciones de test de mT0/mT5
DRIVE_BASE = "/content/drive/MyDrive/chineutro_mt5"

## Drive, repo y datos de Reddit

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_BASE = Path(DRIVE_BASE)
assert DRIVE_BASE.exists(), f"No encuentro {DRIVE_BASE}: ¿montaste el Drive de la cuenta con la que entrenaste?"
print("Carpetas en Drive:", sorted(p.name for p in DRIVE_BASE.iterdir()))

RAIZ = Path("/content/NLP-ChiNeutro")
if not RAIZ.exists():
    url = REPO_URL
    try:
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass   # repo público: no hace falta token
    r = subprocess.run(["git", "clone", "-q", "-b", RAMA, url, str(RAIZ)], capture_output=True, text=True)
    assert r.returncode == 0, f"No pude clonar el repo: {r.stderr[-300:]}"
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "models/gemma4"))
os.chdir(RAIZ)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

from comun import INSTRUCCION_CHAT, norm      # mismo prompt que el entrenamiento de mT0/mT5
from evaluacion import evaluar, cargar_split

In [ ]:
import shutil
import pandas as pd
from google.colab import files

# reddit_chile.json no está en GitHub: se lee desde Drive o se sube a mano (y se guarda en Drive)
RUTA_REDDIT = DRIVE_BASE / "reddit_chile.json"
if not RUTA_REDDIT.exists():
    print(f"No está {RUTA_REDDIT}. Sube data/evaluacion/reddit_chile.json desde tu computador:")
    subido = files.upload()
    assert "reddit_chile.json" in subido, f"Se esperaba reddit_chile.json, llegó {list(subido)}"
    shutil.move("reddit_chile.json", RUTA_REDDIT)
    print(f"Copia guardada en {RUTA_REDDIT}")

datos = json.loads(RUTA_REDDIT.read_text(encoding="utf-8"))
frases = datos["frases"]
print("Fuente:", datos["meta"]["fuente"], "| extracción:", datos["meta"]["fecha_extraccion"],
      "| tope:", datos["meta"]["parametros"]["tope"])
reddit = pd.DataFrame({"id": [f["id"] for f in frases], "frase_chilena": [f["frase"] for f in frases]})
print(f"{len(reddit)} frases de Reddit")
assert len(reddit) == 1900 and reddit.id.is_unique, \
    "Se esperaban 1.900 frases con id único (versión tope 5, la misma de Gemma): revisa el archivo"
if LIMITE:
    reddit = reddit.head(LIMITE)
    print(f"Prueba de humo: solo {LIMITE} frases")
reddit.sample(5, random_state=0)

## Adaptadores entrenados

`modelo_final` es el mejor checkpoint según la pérdida en val (`load_best_model_at_end=True`). Si no estuviera, se
usa el `best_model_checkpoint` que anotó el Trainer. Si no hay ninguno de los dos, ese modelo hay que reentrenarlo.

In [ ]:
def buscar_adaptador(nombre):
    d = DRIVE_BASE / nombre
    final = d / "modelo_final"
    if (final / "adapter_config.json").exists():
        return final
    estados = sorted((d / "checkpoints").glob("checkpoint-*/trainer_state.json"),
                     key=lambda p: int(p.parent.name.split("-")[1]))
    if estados:
        mejor = json.loads(estados[-1].read_text())["best_model_checkpoint"]
        mejor = d / "checkpoints" / Path(mejor).name
        if (mejor / "adapter_config.json").exists():
            print(f"  {nombre}: sin modelo_final, uso {mejor.name}")
            return mejor
    return None

ADAPTADORES = {}
for nombre, base in MODELOS.items():
    ruta = buscar_adaptador(nombre)
    if ruta is None:
        print(f"FALTA {nombre}: no hay adaptador en Drive. Reentrénalo con models/mt0_vs_mt5/entrenar_mt5.ipynb")
        continue
    cfg = json.loads((ruta / "adapter_config.json").read_text())
    assert cfg["base_model_name_or_path"] == base, f"{nombre}: el adaptador es de {cfg['base_model_name_or_path']}"
    assert cfg["r"] == 16, f"{nombre}: r={cfg['r']}, se esperaba 16"
    ADAPTADORES[nombre] = ruta
    print(f"OK {nombre}: {ruta}")
assert ADAPTADORES, "No hay ningún adaptador: hay que entrenar"

## Traducción

In [ ]:
import gc, time
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

def entrada(frase):
    return INSTRUCCION_CHAT.format(chilena=str(frase).strip())

def cargar(nombre):
    base = MODELOS[nombre]
    tok = AutoTokenizer.from_pretrained(base)
    dtype = torch.bfloat16 if BF16 else torch.float32       # nunca fp16 con mT5/mT0 (da NaN)
    m = AutoModelForSeq2SeqLM.from_pretrained(base, dtype=dtype)
    m = PeftModel.from_pretrained(m, str(ADAPTADORES[nombre])).merge_and_unload()
    m.config.use_cache = True
    return tok, m.to("cuda").eval()

def traducir(tok, modelo, frases, bs=BS, max_in=MAX_LEN_IN, max_new=MAX_NEW_TOKENS):
    salidas, truncadas = [], 0
    for i in range(0, len(frases), bs):
        textos = [entrada(f) for f in frases[i:i + bs]]
        truncadas += sum(len(ids) > max_in for ids in tok(textos)["input_ids"])
        x = tok(textos, return_tensors="pt", padding=True, truncation=True, max_length=max_in).to("cuda")
        with torch.no_grad():
            out = modelo.generate(input_ids=x["input_ids"], attention_mask=x["attention_mask"],
                                  do_sample=False, num_beams=1, max_new_tokens=max_new)
        salidas += [s.strip().split("\n")[0].strip() for s in tok.batch_decode(out, skip_special_tokens=True)]
    return salidas, truncadas

Antes de traducir Reddit, cada modelo retraduce las primeras 32 frases de test y las compara con las predicciones
de test que están en el repo (`models/mt0_vs_mt5/predicciones/`). Si coinciden, el adaptador cargado es el mismo que
se evaluó en test.

In [ ]:
test = cargar_split("test")
RUTAS = {}
for nombre in ADAPTADORES:
    dir_pred = DRIVE_BASE / nombre / "predicciones" / "reddit"
    dir_pred.mkdir(parents=True, exist_ok=True)
    ruta = dir_pred / f"predicciones_{nombre}_reddit{'_prueba' if LIMITE else ''}.csv"
    RUTAS[nombre] = ruta
    if ruta.exists() and not REHACER:
        print(f"== {nombre}: ya existe {ruta.name}, lo reutilizo")
        continue

    print(f"== {nombre}")
    t0 = time.time()
    tok, modelo = cargar(nombre)

    # Control: mismas salidas que en test (mismos largos que en el entrenamiento)
    ref = pd.read_csv(RAIZ / f"models/mt0_vs_mt5/predicciones/predicciones_{nombre}_test.csv").head(32)
    rehechas, _ = traducir(tok, modelo, ref.frase_chilena.tolist(), max_in=128, max_new=96)
    iguales = (pd.Series(rehechas) == ref.prediccion.fillna("").str.strip()).mean()
    print(f"  control test: {iguales:.0%} de 32 iguales a las predicciones guardadas")
    if iguales < 0.9:
        print("  OJO: el adaptador no reproduce test. Revisa que sea la corrida correcta antes de seguir.")

    pred, truncadas = traducir(tok, modelo, reddit.frase_chilena.tolist())
    salida = reddit.assign(prediccion=pred)
    salida.to_csv(ruta, index=False)
    vacias = int((salida.prediccion.str.strip() == "").sum())
    copia = (salida.prediccion.map(norm) == salida.frase_chilena.map(norm)).mean()
    print(f"  {ruta.name} | {time.time() - t0:.0f}s | vacías: {vacias} | copia literal: {copia:.1%} "
          f"| entradas truncadas: {truncadas}")
    del modelo
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Ejemplos para revisar a ojo (mismas frases para los dos modelos)
pd.set_option("display.max_colwidth", 150)
ej = reddit.sample(12, random_state=42)[["id", "frase_chilena"]]
for nombre, ruta in RUTAS.items():
    ej = ej.merge(pd.read_csv(ruta)[["id", "prediccion"]].rename(columns={"prediccion": nombre}), on="id")
display(ej.drop(columns="id"))

## Métricas automáticas (sin juez)

Términos chilenos retenidos (sin `clave`, busca todo el diccionario salvo `expresiones_ambiguas.json`) y BERTScore
contra la entrada. BLEU queda en `null` porque no hay referencia. Escribe `_evaluacion.json` y `_por_frase.csv` junto
a cada CSV, que es lo que después completa `juzgar_reddit.py` con el juez.

In [ ]:
RESUMENES = {}
for nombre, ruta in RUTAS.items():
    print(f"== {nombre}")
    RESUMENES[nombre], _ = evaluar(str(ruta), juez=False)
    print(json.dumps(RESUMENES[nombre], ensure_ascii=False, indent=2))

In [ ]:
# Comparación con Gemma 4 en Reddit (resultados ya subidos al repo, con juez)
filas = []
for nombre, r in RESUMENES.items():
    pred = pd.read_csv(RUTAS[nombre])
    filas.append({"modelo": nombre, "n": r["n"], "terminos_retenidos_%": r["terminos_retenidos_%"],
                  "bertscore_f1 (vs entrada)": r.get("bertscore_f1"),
                  "copia_literal_%": round(100 * (pred.prediccion.map(norm) == pred.frase_chilena.map(norm)).mean(), 1),
                  "juez_aceptable_%": None})
for f in sorted((RAIZ / "models/gemma4/predicciones/reddit").glob("*_evaluacion.json")):
    r = json.loads(f.read_text())
    filas.append({"modelo": f.name.replace("predicciones_", "").replace("_reddit_evaluacion.json", ""),
                  "n": r["n"], "terminos_retenidos_%": r["terminos_retenidos_%"],
                  "bertscore_f1 (vs entrada)": r.get("bertscore_f1"), "copia_literal_%": None,
                  "juez_aceptable_%": r.get("juez_aceptable_%")})
pd.DataFrame(filas)

## Descargar resultados

Baja un zip con los CSV de predicciones y sus evaluaciones. Descomprímelo en
`models/mt0_vs_mt5/predicciones/reddit/` del repo en tu computador.

**Ojo:** los `.csv` tienen texto de Reddit y el repo es público. `.gitignore` ya excluye
`models/mt0_vs_mt5/predicciones/reddit/*.csv`, así que en git solo quedan los `_evaluacion.json` (como hizo el grupo
con Gemma). No uses `git add -f` con esos CSV.

In [ ]:
import zipfile
from google.colab import files

zip_ruta = Path("/content/mt0_mt5_reddit.zip")
with zipfile.ZipFile(zip_ruta, "w") as z:
    for nombre, ruta in RUTAS.items():
        for f in sorted(ruta.parent.glob(ruta.stem + "*")):
            z.write(f, f.name)
            print(" ", f.name)
files.download(str(zip_ruta))

## Siguiente paso: juez Opus en tu computador

Desde la raíz del repo, con Claude Code instalado y con sesión iniciada (`claude`, luego `/login`):

```bash
python3 models/gemma4/juzgar_reddit.py \
    models/mt0_vs_mt5/predicciones/reddit/predicciones_mt0_large_lora_r16_reddit.csv \
    models/mt0_vs_mt5/predicciones/reddit/predicciones_mt5_large_lora_r16_reddit.csv
```

Usa la misma rúbrica, effort y caché que Gemma (`evaluacion/juez.py`), y reescribe `_evaluacion.json` y
`_por_frase.csv` con las columnas del juez. Son 3.800 casos (190 lotes de 20): si se acaba la cuota, vuelve a
correrlo más tarde y retoma desde `evaluacion/cache_juez.jsonl`. Para probar antes, agrega `--muestra 100`.